In [2]:
from config import EXTERNAL_DATASET_INPUTS, INTERNAL_DATASET_INPUTS, Columns, ALIGNMENT_PATH
from data_processing.processing import TextProcessor
from data_processing.alignment import Aligner
from utils import stack_csvs_from_folder, drop_empty_rows
import pandas as pd
from data_processing.processing import (
    PREPROCESSING_CONFIG,
    TextProcessor
)
from data_processing.augmentation import DataAugmentation


def load_data() -> pd.DataFrame:
    df_internal = stack_csvs_from_folder(INTERNAL_DATASET_INPUTS)
    return drop_empty_rows(df_internal)


In [3]:
df = load_data()

tp = TextProcessor()

No empty rows found.
Rows before: 1561, Rows after: 1561


In [4]:
df[Columns.TRANSLITERATION] = df[Columns.TRANSLITERATION].apply(lambda x: tp.preprocess_transliteration_text(x, normalize_chars=True, with_hyphens=False))
df[Columns.TRANSLITERATION].head()

0    <lgg_s>kiszib<lgg_e> manubalu2maszur <lgg_s>du...
1    1 <lgg_s>tu2g<lgg_e> sza qa2tim itur4<lgg_s>di...
2    <lgg_s>tu2g<lgg_e> ula idi2nakuum itu3rama 9 <...
3    <lgg_s>kiszib<lgg_e> szu-{d}<lgg_s>en.li2l<lgg...
4    umma szukutumma ana <lgg_s>isztar<lgg_e>la2mas...
Name: transliteration, dtype: str

In [11]:
txt_searched = " manum"

mask = df[Columns.TRANSLITERATION].str.contains(txt_searched, case=False, na=False)

# To see the actual rows:
sarru_rows = df[mask]
print(f"Found {len(sarru_rows)} rows containing '{txt_searched}'")


Found 19 rows containing ' manum'


In [14]:
df[Columns.TRANSLITERATION][mask == True].head(1)

66    1.5 mana <lgg_s>ku3.babbar<lgg_e> iszratim pus...
Name: transliteration, dtype: str

In [ ]:
import re, random, copy, unicodedata
from typing import List, Tuple

def extract_witness_spans(text, pattern, group):
    text = unicodedata.normalize('NFC', text)
    return [(m.start(group), m.end(group), m.group(group))
            for m in pattern.finditer(text)
            if m.group(group)]

def reconstruct(text, spans, shuffled_texts):
    result, cursor = [], 0
    for (start, end, _), new_text in zip(spans, shuffled_texts):
        result.append(text[cursor:start])
        result.append(new_text)
        cursor = end
    result.append(text[cursor:])
    return ''.join(result)

def augment_dataset_witness_shuffle(
    df, akk_col, eng_col, id_col,
    akk_pattern, eng_pattern,
    akk_group=1, eng_group=2,
    n_shuffles=3, verbose_n=5,
):
    augmented = []
    total = skipped = shuffled_count = 0

    for i, (idx, row) in enumerate(df.iterrows()):
        akk = unicodedata.normalize('NFC', str(row[akk_col]))
        eng = unicodedata.normalize('NFC', str(row[eng_col]))
        verbose = (i < verbose_n)

        if not re.match(r'\s*witnessed\s+by', eng, re.IGNORECASE):
            total += 1; skipped += 1
            if verbose: print(f"[{row[id_col]}] SKIP — does not start with 'witnessed by'")
            continue

        akk_spans = extract_witness_spans(akk, akk_pattern, akk_group)
        eng_spans = extract_witness_spans(eng, eng_pattern, eng_group)
        total += 1

        if len(akk_spans) < 2 or len(eng_spans) < 2 or len(akk_spans) != len(eng_spans):
            skipped += 1
            if verbose: print(f"[{row[id_col]}] SKIP — akk={len(akk_spans)} eng={len(eng_spans)} witnesses")
            continue

        if verbose:
            print(f"\n[{row[id_col]}] {len(akk_spans)} witnesses")
            for j, (_, _, t) in enumerate(akk_spans): print(f"  akk[{j}]: {t[:60]}")
            for j, (_, _, t) in enumerate(eng_spans): print(f"  eng[{j}]: {t[:60]}")

        seen = {tuple(range(len(akk_spans)))}
        found = 0
        for _ in range(n_shuffles * 20):
            if found >= n_shuffles: break
            perm = list(range(len(akk_spans)))
            random.shuffle(perm)
            if tuple(perm) in seen: continue
            seen.add(tuple(perm))
            new_akk = reconstruct(akk, akk_spans, [akk_spans[i][2] for i in perm])
            new_eng = reconstruct(eng, eng_spans, [eng_spans[i][2] for i in perm])
            if verbose: print(f"  shuffle {found+1} perm={perm}: {new_eng[:80]}")
            augmented.append({
                id_col: row[id_col], 'row_index': idx,
                akk_col: new_akk, eng_col: new_eng,
                'augmented': True, 'shuffle_idx': found + 1,
            })
            found += 1
        shuffled_count += 1

    print(f"\nTotal: {total} | Skipped: {skipped} | Shuffled: {shuffled_count} | New rows: {len(augmented)}")
    return augmented

# usage — replace with your real patterns
akk_re_wit, eng_re_wit = build_my_patterns()

augmented = augment_dataset_witness_shuffle(
    df_witness,           # your witnessed-by dataframe
    akk_col=Columns.TRANSLITERATION,
    eng_col=Columns.TRANSLATION,
    id_col='id',
    akk_pattern=akk_re_wit, akk_group=1,
    eng_pattern=eng_re_wit, eng_group=2,
    n_shuffles=3,
    verbose_n=5,          # prints details for first 5 rows so you can inspect
)

df_augmented = pd.DataFrame(augmented)